# Cross-Architecture Dimension Change

**Use case:** Map between embedding spaces with **different dimensionalities** — e.g., 384-dim (all-MiniLM-L6) to 1536-dim (OpenAI text-embedding-3-small), or vice versa. This is the hardest case for embedding translation.

**When you'd reach for this:** Your old model and new model have different output dimensions. Not just a rotation — a true change of basis across different-dimensional spaces.

**What you need installed:** `isotrieve`, `numpy`, `scikit-learn`.

**Estimated runtime:** ~2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/AECP/blob/main/isotrieve-python/notebooks/05_cross_architecture_dimension_change.ipynb)

In [ ]:
!pip install -q isotrieve numpy scikit-learn

In [ ]:
import numpy as np
import warnings
from isotrieve.mapping.linear import (
    RidgeMapping,
    OrthogonalProcrustesMapping,
    ProcrustesDiagMapping,
    LowRankAffineMapping,
)
from isotrieve.quality.gate import QualityGate

print("imports OK")

## 1. Which strategies support non-square transforms?

| Strategy | d_src == d_tgt | d_src != d_tgt | Notes |
|----------|:-:|:-:|-------|
| `RidgeMapping` | Yes | **Yes** | Best for rectangular transforms |
| `LowRankAffineMapping` | Yes | **Yes** | Ridge + SVD compression |
| `OrthogonalProcrustesMapping` | Yes | **No** | Raises ValueError |
| `ProcrustesDiagMapping` | Yes | **No** | Raises ValueError |

For cross-architecture (different dims), use **Ridge** or **LowRankAffine**.

## 2. Generate synthetic spaces with different dimensions

In [ ]:
rng = np.random.default_rng(42)

N_CAL = 2000
N_EVAL = 500
LATENT = 32  # shared semantic content

def make_space(n, latent_dim, embed_dim, rng):
    latent = rng.normal(size=(n, latent_dim))
    W = rng.normal(size=(latent_dim, embed_dim)) / np.sqrt(latent_dim)
    vecs = latent @ W
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

# Calibration
latent_cal = rng.normal(size=(N_CAL, LATENT))
W_384 = rng.normal(size=(LATENT, 384)) / np.sqrt(LATENT)
W_1536 = rng.normal(size=(LATENT, 1536)) / np.sqrt(LATENT)

X_cal = (latent_cal @ W_384)
X_cal = X_cal / np.linalg.norm(X_cal, axis=1, keepdims=True)
Y_cal = (latent_cal @ W_1536)
Y_cal = Y_cal / np.linalg.norm(Y_cal, axis=1, keepdims=True)

# Evaluation (disjoint)
latent_eval = rng.normal(size=(N_EVAL, LATENT))
X_eval = (latent_eval @ W_384)
X_eval = X_eval / np.linalg.norm(X_eval, axis=1, keepdims=True)
Y_eval = (latent_eval @ W_1536)
Y_eval = Y_eval / np.linalg.norm(Y_eval, axis=1, keepdims=True)

print(f"Source space: {X_cal.shape[1]} dims")
print(f"Target space: {Y_cal.shape[1]} dims")
print(f"Dimension ratio: {Y_cal.shape[1] / X_cal.shape[1]:.1f}x")

## 3. Fit RidgeMapping (supports rectangular)

In [ ]:
ridge = RidgeMapping(alpha="auto", seed=0)
ridge.fit(X_cal, Y_cal)

vr = ridge.validation_report()
print(f"Ridge: {ridge.d_src} -> {ridge.d_tgt}")
print(f"  Alpha: {vr.alpha:.2f}")
print(f"  Holdout cosine mean: {vr.holdout_cosine_mean:.4f}")
print(f"  Top-1 retention: {vr.top1_retention:.4f}")
print(f"  Top-10 retention: {vr.top10_retention:.4f}")

## 4. Fit LowRankAffineMapping (rectangular + compression)

In [ ]:
lowrank = LowRankAffineMapping(alpha="auto", rank=128, seed=0)
lowrank.fit(X_cal, Y_cal)

vr_lr = lowrank.validation_report()
print(f"LowRankAffine (rank=128): {lowrank.d_src} -> {lowrank.d_tgt}")
print(f"  Alpha: {vr_lr.alpha:.2f}")
print(f"  Holdout cosine mean: {vr_lr.holdout_cosine_mean:.4f}")
print(f"  Top-1 retention: {vr_lr.top1_retention:.4f}")

## 5. Confirm Procrustes rejects rectangular transforms

In [ ]:
try:
    proc = OrthogonalProcrustesMapping(seed=0)
    proc.fit(X_cal, Y_cal)
    print("ERROR: Should have raised ValueError")
except ValueError as e:
    print(f"Correctly rejected: {e}")

try:
    proc_diag = ProcrustesDiagMapping(seed=0)
    proc_diag.fit(X_cal, Y_cal)
    print("ERROR: Should have raised ValueError")
except ValueError as e:
    print(f"Correctly rejected: {e}")

## 6. Evaluate on held-out data

In [ ]:
gate = QualityGate()

# Ridge gate
gr = gate.evaluate(ridge, X_eval, Y_eval)
print(f"Ridge gate: {gr.verdict.value}")
print(f"  Predicted retention: {gr.predicted_retention:.3f}")
print(f"  Cosine mean: {gr.cosine_mean:.4f}")
print(f"  Top-1: {gr.top1_retention:.4f}, Top-10: {gr.top10_retention:.4f}")

print()

# LowRank gate
gr_lr = gate.evaluate(lowrank, X_eval, Y_eval)
print(f"LowRank gate: {gr_lr.verdict.value}")
print(f"  Predicted retention: {gr_lr.predicted_retention:.3f}")
print(f"  Cosine mean: {gr_lr.cosine_mean:.4f}")
print(f"  Top-1: {gr_lr.top1_retention:.4f}, Top-10: {gr_lr.top10_retention:.4f}")

## 7. How does quality degrade as the dimension gap widens?

In [ ]:
target_dims = [384, 512, 768, 1024, 1536]
results = []

for d_tgt in target_dims:
    rng2 = np.random.default_rng(42)
    latent2 = rng2.normal(size=(N_CAL, LATENT))
    W_t = rng2.normal(size=(LATENT, d_tgt)) / np.sqrt(LATENT)
    Y2 = (latent2 @ W_t)
    Y2 = Y2 / np.linalg.norm(Y2, axis=1, keepdims=True)
    
    Y_eval2 = (rng2.normal(size=(N_EVAL, LATENT)) @ W_t)
    Y_eval2 = Y_eval2 / np.linalg.norm(Y_eval2, axis=1, keepdims=True)
    
    m = RidgeMapping(alpha="auto", seed=0)
    m.fit(X_cal, Y2)
    vr = m.validation_report()
    results.append({
        "d_tgt": d_tgt,
        "ratio": d_tgt / 384,
        "cosine_mean": vr.holdout_cosine_mean,
        "top1": vr.top1_retention,
        "top10": vr.top10_retention,
    })

print(f"{'d_tgt':>6} {'ratio':>6} {'cosine':>8} {'top1':>8} {'top10':>8}")
print("-" * 40)
for r in results:
    print(f"{r['d_tgt']:>6} {r['ratio']:>6.1f}x {r['cosine_mean']:>8.4f} {r['top1']:>8.4f} {r['top10']:>8.4f}")

## Summary

- **RidgeMapping** handles any dimension combination (384→1536, 1536→384, etc.)
- **LowRankAffineMapping** adds SVD compression for smaller file sizes
- **Procrustes variants** require square dims — use them when d_src == d_tgt
- Quality generally degrades as the dimension ratio increases, but stays usable for moderate ratios

## Try it yourself

Try `d_tgt = 128` (downscaling from 384). Does downscaling work better or worse than upscaling? Why?

```python
d_tgt = 128  # downscaling
```